# 書籍公開記念記事
## Sentinel-2 データの処理と可視化

記事リンク: []()

## 概要

Sentinel-2 衛星データの取得とバンド処理を行います。


## 使用データ

| 項目 | 情報 |
| ---- | ---- | 
| 衛星 | Sentinel-2 |
| 処理レベル | L2A |
| 画像クレジット| ©ESA Copernicus Sentinel-2 |

## Sentinel-2

In [ ]:
import os
import warnings
from glob import glob
import pandas as pd

import pystac_client
import requests

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from tqdm import tqdm

PATH_OUTPUT = os.path.join('output', '3_3_3_999')
os.makedirs(PATH_OUTPUT, exist_ok=True)

import fiona
import rasterio
import rasterio.mask

## Crop Areas

In [ ]:
bbox = [14.34770, 50.17473, 14.49809,50.26115,]

In [10]:
# STAC APIのURLを指定
API_URL = "https://earth-search.aws.element84.com/v1"

# クライアントを作成
client = pystac_client.Client.open(API_URL)

# 検索条件を設定 (例: 時間範囲とエリア、雲量フィルタリング)

search = client.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2020-03-01T00:00:00Z/2020-10-30T23:59:59Z",
    query={"eo:cloud_cover": {"lt": 40}},  # 雲量が10%未満の画像をフィルタリング
    max_items=40,
    limit=100,
)

# 検索結果を取得
items = search.item_collection()
print(f"Found {len(items)} items")

# 完全なタイルをフィルタリングする関数
def is_tile_full(item, bbox):
    # タイルの境界ボックスを取得
    item_bbox = item.bbox
    # bboxが完全にタイル内に収まっているかチェック
    return (bbox[0] >= item_bbox[0] and bbox[2] <= item_bbox[2] and
            bbox[1] >= item_bbox[1] and bbox[3] <= item_bbox[3])

# フィルタリングされたアイテムを保持するリスト
filtered_items = [item for item in items if is_tile_full(item, bbox)]
print(f"Found {len(filtered_items)} full coverage items")


# アイテムをループして処理
for item in filtered_items:
    # アイテムのIDとアセット情報を取得
    item_id = item.id
    assets = item.assets
    print(f"Processing {item_id}",)
    
    # 可視化用のバンドを指定 (B04, B03, B02: 赤, 緑, 青)
    bands = [ "blue", "green","red","nir", "scl"]
    band_files = []

    for band in bands:
        # アセットのURLを取得
        asset = assets.get(f"{band}")
        if asset is None:
            continue
        url = asset.href
        print(f"Downloading {band} from {url}")

        # ファイルをダウンロードして保存
        band_file = os.path.join(PATH_OUTPUT, f"{item_id}_{band}.tiff")
        response = requests.get(url)
        with open(band_file, 'wb') as f:
            f.write(response.content)
        band_files.append(band_file)

print("Download and visualization completed.")


Found 40 items
Found 40 full coverage items
Processing S2A_33UVR_20201025_0_L2A
Processing S2A_33UVR_20201025_1_L2A
Processing S2A_33UVR_20201012_1_L2A
Processing S2A_33UVR_20201012_0_L2A
Processing S2A_33UVR_20201002_0_L2A
Processing S2B_33UVR_20200930_1_L2A
Processing S2B_33UVR_20200930_0_L2A
Processing S2A_33UVR_20200922_0_L2A
Processing S2A_33UVR_20200922_1_L2A
Processing S2B_33UVR_20200920_0_L2A
Processing S2B_33UVR_20200920_1_L2A
Processing S2A_33UVR_20200915_1_L2A
Processing S2A_33UVR_20200915_0_L2A
Processing S2A_33UVR_20200912_1_L2A
Processing S2A_33UVR_20200912_0_L2A
Processing S2B_33UVR_20200907_1_L2A
Processing S2B_33UVR_20200907_0_L2A
Processing S2B_33UVR_20200828_1_L2A
Processing S2B_33UVR_20200828_0_L2A
Processing S2A_33UVR_20200826_1_L2A
Processing S2A_33UVR_20200826_0_L2A
Processing S2B_33UVR_20200821_1_L2A
Processing S2B_33UVR_20200821_0_L2A
Processing S2A_33UVR_20200816_1_L2A
Processing S2A_33UVR_20200816_0_L2A
Processing S2A_33UVR_20200813_1_L2A
Processing S2A_33UVR

In [11]:
from osgeo import gdal, ogr

def merge_to_rgb(red_tiff, green_tiff, blue_tiff, output_tiff):
    # Open the source datasets
    red_ds = gdal.Open(red_tiff)
    green_ds = gdal.Open(green_tiff)
    blue_ds = gdal.Open(blue_tiff)

    if red_ds is None or green_ds is None or blue_ds is None:
        raise FileNotFoundError("One or more input files could not be opened")

    # Get the width, height, and projection from the red band
    width = red_ds.RasterXSize
    height = red_ds.RasterYSize
    projection = red_ds.GetProjection()
    geotransform = red_ds.GetGeoTransform()

    # Create the output dataset
    driver = gdal.GetDriverByName('GTiff')
    output_ds = driver.Create(output_tiff, width, height, 3)

    if output_ds is None:
        raise RuntimeError(f"Unable to create {output_tiff}")

    # Set the projection and geotransform
    output_ds.SetProjection(projection)
    output_ds.SetGeoTransform(geotransform)

    # Read the data from each band
    red_band = red_ds.GetRasterBand(1).ReadAsArray()
    green_band = green_ds.GetRasterBand(1).ReadAsArray()
    blue_band = blue_ds.GetRasterBand(1).ReadAsArray()

    # Write the data to the output dataset
    output_ds.GetRasterBand(1).WriteArray(red_band)
    output_ds.GetRasterBand(2).WriteArray(green_band)
    output_ds.GetRasterBand(3).WriteArray(blue_band)

    # Close the datasets
    red_ds = None
    green_ds = None
    blue_ds = None
    output_ds = None

# Usage
red_tiff = os.path.join(PATH_OUTPUT, f"{item_id}_red.tiff")
green_tiff = os.path.join(PATH_OUTPUT, f"{item_id}_green.tiff")
blue_tiff = os.path.join(PATH_OUTPUT, f"{item_id}_blue.tiff")
output_tiff = os.path.join(PATH_OUTPUT, f"{item_id}_rgb.tiff")

# merge_to_rgb(red_tiff, green_tiff, blue_tiff, output_tiff)
!gdal_merge.py -separate {blue_tiff} {green_tiff} {red_tiff} -o {output_tiff}

0Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
...10...20...30...40...50...60...70...80...90...100 - done.


In [12]:
item.assets

{'aot': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/33/U/VR/2020/7/S2B_33UVR_20200722_1_L2A/AOT.tif>,
 'blue': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/33/U/VR/2020/7/S2B_33UVR_20200722_1_L2A/B02.tif>,
 'coastal': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/33/U/VR/2020/7/S2B_33UVR_20200722_1_L2A/B01.tif>,
 'granule_metadata': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/33/U/VR/2020/7/S2B_33UVR_20200722_1_L2A/granule_metadata.xml>,
 'green': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/33/U/VR/2020/7/S2B_33UVR_20200722_1_L2A/B03.tif>,
 'nir': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/33/U/VR/2020/7/S2B_33UVR_20200722_1_L2A/B08.tif>,
 'nir08': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/33/U/VR/2020/7/S2B_33UVR_20200722_1

In [14]:
# STAC APIのURLを指定
API_URL = "https://earth-search.aws.element84.com/v1"

# クライアントを作成
client = pystac_client.Client.open(API_URL)

# 検索条件を設定 (例: 時間範囲とエリア、雲量フィルタリング)

search = client.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2020-03-01T00:00:00Z/2020-07-30T23:59:59Z",
    query={"eo:cloud_cover": {"lt": 40}},  # 雲量が10%未満の画像をフィルタリング
    max_items=80,
    limit=100,
)

# 検索結果を取得
items = search.item_collection()
print(f"Found {len(items)} items")

# 完全なタイルをフィルタリングする関数
def is_tile_full(item, bbox):
    # タイルの境界ボックスを取得
    item_bbox = item.bbox
    # bboxが完全にタイル内に収まっているかチェック
    return (bbox[0] >= item_bbox[0] and bbox[2] <= item_bbox[2] and
            bbox[1] >= item_bbox[1] and bbox[3] <= item_bbox[3])

# フィルタリングされたアイテムを保持するリスト
filtered_items = [item for item in items if is_tile_full(item, bbox)]
print(f"Found {len(filtered_items)} full coverage items")


# アイテムをループして処理
for item in filtered_items:
    # アイテムのIDとアセット情報を取得
    item_id = item.id
    assets = item.assets
    print(f"Processing {item_id}",)
    
    # 可視化用のバンドを指定 (B04, B03, B02: 赤, 緑, 青)
    bands = [ "blue", "green","red","nir", "scl"]
    band_files = []

    for band in bands:
        # アセットのURLを取得
        asset = assets.get(f"{band}")
        if asset is None:
            continue
        url = asset.href
        print(f"Downloading {band} from {url}")

        # ファイルをダウンロードして保存
        band_file = os.path.join(PATH_OUTPUT, f"{item_id}_{band}.tiff")
        response = requests.get(url)
        with open(band_file, 'wb') as f:
            f.write(response.content)
        band_files.append(band_file)

print("Download and visualization completed.")


Found 42 items
Found 42 full coverage items
Processing S2B_33UVR_20200729_1_L2A
Processing S2B_33UVR_20200729_0_L2A
Processing S2A_33UVR_20200727_0_L2A
Processing S2A_33UVR_20200724_0_L2A
Processing S2B_33UVR_20200722_1_L2A
Processing S2B_33UVR_20200722_0_L2A
Processing S2B_33UVR_20200719_1_L2A
Processing S2B_33UVR_20200719_0_L2A
Processing S2A_33UVR_20200714_1_L2A
Processing S2A_33UVR_20200714_0_L2A
Processing S2A_33UVR_20200704_1_L2A
Processing S2A_33UVR_20200704_0_L2A
Processing S2A_33UVR_20200627_1_L2A
Processing S2A_33UVR_20200627_0_L2A
Processing S2B_33UVR_20200622_1_L2A
Processing S2B_33UVR_20200622_0_L2A
Processing S2B_33UVR_20200612_1_L2A
Processing S2B_33UVR_20200612_0_L2A
Processing S2B_33UVR_20200530_0_L2A
Processing S2A_33UVR_20200518_1_L2A
Processing S2A_33UVR_20200518_0_L2A
Processing S2A_33UVR_20200508_3_L2A
Processing S2A_33UVR_20200508_1_L2A
Processing S2A_33UVR_20200428_1_L2A
Processing S2A_33UVR_20200428_0_L2A
Processing S2B_33UVR_20200423_1_L2A
Processing S2B_33UVR

In [4]:
PATH_ROOT = os.path.join('output', '203',)
PATHS_S2 = sorted(glob(os.path.join(PATH_ROOT, 'S2*.tiff')))

df = pd.DataFrame(PATHS_S2, columns=['path'])
df['name'] = df['path'].apply(lambda x: os.path.basename(x).split('.')[0])
df['band'] = df['name'].apply(lambda x: x.split('_')[-1])
df['date'] = df['name'].apply(lambda x: x.split('_')[2][:8])

# date time type
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

print(f'NUM Sentinel-2 AREA: {len(df)}')
df.head()

NUM Sentinel-2 AREA: 373


,path,name,band,date
0,output/203/S2A_33UVR_20200329_0_L2A_blue.tiff,S2A_33UVR_20200329_0_L2A_blue,blue,2020-03-29
1,output/203/S2A_33UVR_20200329_0_L2A_green.tiff,S2A_33UVR_20200329_0_L2A_green,green,2020-03-29
2,output/203/S2A_33UVR_20200329_0_L2A_nir.tiff,S2A_33UVR_20200329_0_L2A_nir,nir,2020-03-29
3,output/203/S2A_33UVR_20200329_0_L2A_red.tiff,S2A_33UVR_20200329_0_L2A_red,red,2020-03-29
4,output/203/S2A_33UVR_20200329_0_L2A_scl.tiff,S2A_33UVR_20200329_0_L2A_scl,scl,2020-03-29


In [5]:
area_of_interests = [
    f'crop_{str(iii+1).zfill(3)}'
    for iii in range(8)
] + ['airport', 'tree']

PATHS_GEOJSON = [
    os.path.join('../data/paz/polygon', f'{area}.geojson')
    for area in area_of_interests
]

# area crop process
for idx_tif, PATH_S2 in tqdm(enumerate(df['path']), total=len(df)):
    # print(f'PROCESSING: {idx_tif+1}/{len(PATHS_PAZ)}')
    file_name_tif = os.path.basename(PATH_S2).split('.')[0]
    
    PATH_S2_EPSG4326 = os.path.join(PATH_ROOT, f'{file_name_tif}_EPSG4326.tif')
    !gdalwarp -q -t_srs EPSG:4326 {PATH_S2} {PATH_S2_EPSG4326}
    
    for idx_geojson, PATH_AREA_GEOJSON in enumerate(PATHS_GEOJSON):
        # polygon 
        with fiona.open(PATH_AREA_GEOJSON, "r") as shapefile:
            shapes = [feature["geometry"] for feature in shapefile]
        file_name_geojson = os.path.basename(PATH_AREA_GEOJSON).split('.')[0]
        
        # crop 
        with rasterio.open(PATH_S2_EPSG4326) as src:
            out_image, out_transform = rasterio.mask.mask(src, shapes, 
                crop=True, nodata=-1, filled=True)
            out_meta = src.meta

        # write output
        out_meta.update({"driver": "GTiff",
                        "height": out_image.shape[1],
                        "width": out_image.shape[2],
                        "transform": out_transform,})
        PATH_OUTPUT_CROP = os.path.join(PATH_ROOT, 'crop',
                                        f'{file_name_tif}_{file_name_geojson}.tif')
        os.makedirs(os.path.dirname(PATH_OUTPUT_CROP), exist_ok=True)
        with rasterio.open(PATH_OUTPUT_CROP, "w", **out_meta) as dest:
            dest.write(out_image)
            
    !rm {PATH_S2_EPSG4326}

100%|██████████| 373/373 [18:29<00:00,  2.97s/it]


In [10]:
PATH_ROOT = os.path.join('output', '203','crop')
PATHS_S2 = sorted(glob(os.path.join(PATH_ROOT, 'S2*.tif')))

df = pd.DataFrame(PATHS_S2, columns=['path'])
df['name'] = df['path'].apply(lambda x: os.path.basename(x).split('.')[0])
df['band'] = df['name'].apply(lambda x: x.split('_')[5])
df['target'] = df['name'].apply(lambda x: '_'.join(x.split('_')[6:8]))
df['date'] = df['name'].apply(lambda x: x.split('_')[2][:8])

# date time type
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

print(f'NUM Sentinel-2 AREA: {len(df)}')
df.head()

NUM Sentinel-2 AREA: 3730


,path,name,band,target,date
0,output/203/crop/S2A_33UVR_20200329_0_L2A_blue_...,S2A_33UVR_20200329_0_L2A_blue_airport,blue,airport,2020-03-29
1,output/203/crop/S2A_33UVR_20200329_0_L2A_blue_...,S2A_33UVR_20200329_0_L2A_blue_crop_001,blue,crop_001,2020-03-29
2,output/203/crop/S2A_33UVR_20200329_0_L2A_blue_...,S2A_33UVR_20200329_0_L2A_blue_crop_002,blue,crop_002,2020-03-29
3,output/203/crop/S2A_33UVR_20200329_0_L2A_blue_...,S2A_33UVR_20200329_0_L2A_blue_crop_003,blue,crop_003,2020-03-29
4,output/203/crop/S2A_33UVR_20200329_0_L2A_blue_...,S2A_33UVR_20200329_0_L2A_blue_crop_004,blue,crop_004,2020-03-29


In [34]:
from osgeo import gdal

def merge_rasters(input_files, output_file):
    # 入力ファイルをリスト化
    src_files_to_mosaic = [gdal.Open(f) for f in input_files]
    
    # 最初のラスターのプロパティを取得
    first_raster = src_files_to_mosaic[0]
    output_driver = gdal.GetDriverByName("GTiff")
    
    # 出力する結合されたラスターの幅と高さを計算
    width = max([raster.RasterXSize for raster in src_files_to_mosaic])
    height = max([raster.RasterYSize for raster in src_files_to_mosaic])
    
    # 出力ファイルの作成
    output_raster = output_driver.Create(
        output_file, width, height, first_raster.RasterCount, 
        first_raster.GetRasterBand(1).DataType)
    
    # 出力ファイルのジオトランスフォームと投影を設定
    output_raster.SetGeoTransform(first_raster.GetGeoTransform())
    output_raster.SetProjection(first_raster.GetProjection())
    
    # 各入力ファイルを出力ファイルに配置
    x_offset = 0
    for raster in src_files_to_mosaic:
        for band_index in range(1, raster.RasterCount + 1):
            input_band = raster.GetRasterBand(band_index)
            output_band = output_raster.GetRasterBand(band_index)
            data = input_band.ReadAsArray()
            output_band.WriteArray(data, xoff=x_offset)
        x_offset += raster.RasterXSize  # X方向にオフセットを追加
        
    # 出力ファイルを保存
    output_raster.FlushCache()



def merge_rasters_as_bands(input_files, output_file):
    # 最初のラスターを基準にしてサイズ、ジオトランスフォーム、プロジェクションを取得
    first_raster = gdal.Open(input_files[0])
    width = first_raster.RasterXSize
    height = first_raster.RasterYSize
    geotransform = first_raster.GetGeoTransform()
    projection = first_raster.GetProjection()
    data_type = first_raster.GetRasterBand(1).DataType

    # 出力ファイルのドライバを指定し、バンド数を指定して新規作成
    driver = gdal.GetDriverByName("GTiff")
    output_raster = driver.Create(output_file, width, height, len(input_files), data_type)
    output_raster.SetGeoTransform(geotransform)
    output_raster.SetProjection(projection)
    
    # 各入力ファイルをリサンプリングして出力ファイルの異なるバンドとして追加
    for i, file in enumerate(input_files):
        src_raster = gdal.Open(file)
        
        # 入力ラスターを最初のラスターと同じサイズにリサンプリング
        resampled_raster = gdal.Warp(
            '', src_raster, format='MEM', width=width, height=height,
            resampleAlg=0
        )
        
        # リサンプリングしたデータを読み込んで、出力ファイルのバンドに書き込み
        band_data = resampled_raster.GetRasterBand(1).ReadAsArray()
        output_band = output_raster.GetRasterBand(i + 1)
        output_band.WriteArray(band_data)

    # データをディスクにフラッシュ
    output_raster.FlushCache()
    output_raster = None  # ファイルを閉じる



In [35]:
os.makedirs(os.path.join(PATH_ROOT, 'merge'), exist_ok=True)
for blue, green, red, nir, scl in tqdm(zip(
    df[df['band']=='blue']['path'].values,
    df[df['band']=='green']['path'].values,
    df[df['band']=='red']['path'].values,
    df[df['band']=='nir']['path'].values,
    df[df['band']=='scl']['path'].values,
), total=len(df[df['band']=='blue'])):
    file_name = "_".join(os.path.basename(blue).split('_')[0:5])
    area = "_".join(os.path.basename(blue).split('_')[6:8])
    # print(file_name, area)
    output_file = os.path.join(PATH_ROOT, 'merge', f'{file_name}_merge_{area}')
    merge_rasters_as_bands([blue, green, red, nir, scl], output_file)
    # break

 99%|█████████▊| 740/750 [00:05<00:00, 124.49it/s]
